In [ ]:
from pathlib import Path
import glob
import os
import re

import numpy as np
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
import pandas as pd

In [ ]:
# Input data paths
country_thresholds_path = "../data/country_thresholds.csv"
wb_income_classification_path = "../data/wb_income_classification.csv"
results_dir = "/data/tc-grid/open-gira/results"

# Output paths
plot_dir = Path("../plots")

# Do not extrapolate beyond the available events
tc_model_n_years = {
    "CHAZ": 1000,
    "MIT": 200,
    "IRIS": 1000,
    "STORM": 1000,
}

# Mapping from open-gira STORM_SET to display name
atmospheres_to_scenario = {
    "STORM_constant": "2000 STORM",
    "STORM_CNRM-CM6-1-HR": "2050 STORM RCP8.5 CNRM",
    "STORM_CMCC-CM2-VHR4": "2050 STORM RCP8.5 CMCC",
    "STORM_EC-Earth3P-HR": "2050 STORM RCP8.5 EC-Earth",
    "STORM_HadGEM3-GC31-HM": "2050 STORM RCP8.5 HadGEM3",
    "IRIS_PRESENT": "2000 IRIS",
    "IRIS_SSP5-2050": "2050 IRIS SSP5-8.5",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2010": "2010 CHAZ UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2010": "2010 CHAZ CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2010": "2010 CHAZ CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2010": "2010 CHAZ EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2010": "2010 CHAZ MIROC6",
    "CHAZ_SSP-585_GCM-UKESM1-0-LL_epoch-2050": "2050 CHAZ SSP5-8.5 UKESM1",
    "CHAZ_SSP-585_GCM-CESM2_epoch-2050": "2050 CHAZ SSP5-8.5 CESM2",
    "CHAZ_SSP-585_GCM-CNRM-CM6-1_epoch-2050": "2050 CHAZ SSP5-8.5 CNRM",
    "CHAZ_SSP-585_GCM-EC-Earth3_epoch-2050": "2050 CHAZ SSP5-8.5 EC-Earth3",
    "CHAZ_SSP-585_GCM-MIROC6_epoch-2050": "2050 CHAZ SSP5-8.5 MIROC6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2005": "2005 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2005": "2005 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2005": "2005 Emanuel EC-Earth6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2005": "2005 Emanuel MIROC6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2005": "2005 Emanuel UKMO6",
    "emanuel_ssp-585_gcm-cesm2_epoch-2050": "2050 Emanuel CESM2",
    "emanuel_ssp-585_gcm-cnrm6_epoch-2050": "2050 Emanuel CNRM6",
    "emanuel_ssp-585_gcm-ecearth6_epoch-2050": "2050 Emanuel EC-Earth6",
    "emanuel_ssp-585_gcm-miroc6_epoch-2050": "2050 Emanuel MIROC6",
    "emanuel_ssp-585_gcm-ukmo6_epoch-2050": "2050 Emanuel UKMO6",
}
# Specify group members (GCM variants) of track sets
gcm_groups = {
    "IRIS baseline": ["2000 IRIS"],
    "IRIS SSP5-8.5 2050": ["2050 IRIS SSP5-8.5"],
    "STORM baseline": ["2000 STORM"],
    "STORM RCP8.5 2050": [
        "2050 STORM RCP8.5 CNRM",
        "2050 STORM RCP8.5 CMCC",
        "2050 STORM RCP8.5 EC-Earth",
        "2050 STORM RCP8.5 HadGEM3"
    ],
    "CHAZ baseline": [
        "2010 CHAZ UKESM1",
        "2010 CHAZ CESM2",
        "2010 CHAZ CNRM",
        "2010 CHAZ EC-Earth3",
        "2010 CHAZ MIROC6",
    ],
    "CHAZ SSP5-8.5 2050": [
        "2050 CHAZ SSP5-8.5 UKESM1",
        "2050 CHAZ SSP5-8.5 CESM2",
        "2050 CHAZ SSP5-8.5 CNRM",
        "2050 CHAZ SSP5-8.5 EC-Earth3",
        "2050 CHAZ SSP5-8.5 MIROC6",
    ],   
    "MIT baseline": [
        "2005 Emanuel CESM2",
        "2005 Emanuel CNRM6",
        "2005 Emanuel EC-Earth6",
        "2005 Emanuel MIROC6",
        "2005 Emanuel UKMO6",
    ],
    "MIT SSP5-8.5 2050": [
        "2050 Emanuel CESM2",
        "2050 Emanuel CNRM6",
        "2050 Emanuel EC-Earth6",
        "2050 Emanuel MIROC6",
        "2050 Emanuel UKMO6",
    ],
}
group_colours = {
    "IRIS": "lightgreen",
    "STORM": "plum",
    "CHAZ": "pink",
    "MIT": "lightblue",
}

# Assemble optimum country to threshold mapping
# Calibrated thresholds, ms-1
per_country_calibrated_thresholds = pd.read_csv(country_thresholds_path).rename(columns={"iso_a3": "ISO_A3"}).set_index("ISO_A3")
# Thresholds based on income, ms-1
income_threshold_map = {"L": "30.0", "LM": "30.0", "UM": "30.0", "H": "32.5"}
thresholds = pd.read_csv(wb_income_classification_path, comment="#").set_index("ISO_A3")
thresholds["threshold_ms-1"] = thresholds.INCOME_LEVEL.map(income_threshold_map)
thresholds.loc[per_country_calibrated_thresholds.index, "threshold_ms-1"] = per_country_calibrated_thresholds["threshold_ms-1"].astype(str)

#rps = np.array([i * base for base in (1, 10, 100) for i in range(1, 10)])
rps = np.logspace(0, 3, 40)

In [ ]:
def compute_country_pop_affected_by_rp(
    atmospheres_to_scenario,
    gcm_groups,
    tc_model_n_years,
    thresholds,
    results_dir,
    rps,
):
    """
    Build country-level pop_affected curves interpolated onto a common
    return_period_years grid, keeping TC model / Scenario / Country
    information intact.

    Returns
    -------
    baseline, future : pd.DataFrame
        Each indexed by MultiIndex (TC model, Scenario, Country,
        return_period_years) with a single column 'pop_affected'.
    """
    baseline_frames = []
    future_frames = []

    for atmosphere, scenario in atmospheres_to_scenario.items():

        print(atmosphere, end="\r")
        paths = glob.glob(os.path.join(
            results_dir, f"power/by_country/*/disruption/{atmosphere}/pop_affected_by_event.pq"
        ))
        data = {path.split("/")[-4]: pd.read_parquet(path) for path in paths}

        tc_model = atmosphere.split("_")[0]
        if tc_model == "emanuel":
            tc_model = "MIT"
        years_of_events = tc_model_n_years[tc_model]

        # Work out which epoch (baseline/future) this scenario belongs to
        for group_name, members in gcm_groups.items():
            if scenario in members:
                break
        family, *other = group_name.split()
        epoch = "baseline" if other[0] == "baseline" else "future"

        for iso_a3, df in data.items():
            try:
                threshold = thresholds.loc[iso_a3, "threshold_ms-1"]
                series = df.loc[:, str(threshold)]
            except KeyError:
                continue

            cdf = series.rename("pop_affected").reset_index()

            if atmosphere.startswith("STORM") or atmosphere.startswith("IRIS"):
                cdf["year"] = cdf.event_id.apply(lambda s: int(s.split("_")[2]))
            elif atmosphere.startswith("CHAZ") or atmosphere.startswith("emanuel"):
                cdf["year"] = cdf.event_id.apply(lambda s: int(re.search(r"Y(\d{4})", s).groups()[0]))

            annual_sum = cdf.drop(columns=["event_id"]).groupby("year")["pop_affected"].sum()

            # Use the known simulation length rather than inferring it from
            # this country's data -- a country can have zero affected
            # population (and therefore no rows at all) in many years.
            n_years = years_of_events
            full_years = pd.RangeIndex(0, n_years, name="year")
            annual_sum = annual_sum.reindex(full_years, fill_value=0.0)

            cdf = annual_sum.reset_index().sort_values("pop_affected").reset_index(drop=True)

            year_range = range(n_years - 1, -1, -1)
            cdf["rank"] = year_range
            cdf["return_period_years"] = (n_years + 1) / cdf["rank"]

            cdf = cdf.drop(columns=["year", "rank"])
            cdf = cdf[np.isfinite(cdf.return_period_years)]
            cdf["log_rp"] = np.log(cdf.return_period_years)

            target = pd.DataFrame({
                "return_period_years": rps,
                "log_rp": np.log(rps),
                "pop_affected": np.nan,
            })
            cdf = pd.concat([cdf, target]).sort_values("log_rp").set_index("log_rp")
            cdf = cdf.interpolate(method="index")
            cdf = cdf.loc[np.log(rps)].reset_index(drop=True)

            cdf.loc[cdf.return_period_years > years_of_events, "pop_affected"] = np.nan

            cdf["TC model"] = tc_model
            cdf["Scenario"] = scenario
            cdf["Country"] = iso_a3

            cdf = cdf.set_index(["TC model", "Scenario", "Country", "return_period_years"])

            if epoch == "baseline":
                baseline_frames.append(cdf)
            else:
                future_frames.append(cdf)

    baseline = pd.concat(baseline_frames) if baseline_frames else pd.DataFrame()
    future = pd.concat(future_frames) if future_frames else pd.DataFrame()

    return baseline, future


def collapse_scenario_mean(df):
    """
    Collapse the 'Scenario' level of the MultiIndex by averaging, leaving
    one value per (TC model, Country, return_period_years).
    """
    return df.groupby(level=["TC model", "Country", "return_period_years"]).mean()

baseline, future = compute_country_pop_affected_by_rp(
    atmospheres_to_scenario, gcm_groups, tc_model_n_years, thresholds, results_dir, rps
)

baseline_mean = collapse_scenario_mean(baseline)
future_mean = collapse_scenario_mean(future)

df = future_mean.rename(columns={"pop_affected": "future"}).join(baseline_mean.rename(columns={"pop_affected": "baseline"}))
df["ratio"] = df.future / df.baseline
df["diff"] = df.future - df.baseline

In [ ]:
# Largest increases in risk across TC model / country

f, ax = plt.subplots(figsize=(6, 4))
top_n = 10
rp = 50
to_plot = (
    df.loc[:, :, rp - 1: rp + 1, :]
        .droplevel("return_period_years")
        .sort_values("diff", ascending=False)["diff"]
        .iloc[:top_n]
        .iloc[::-1]
)
to_plot.plot(kind="barh", ax=ax)
ax.grid()

In [ ]:
# Largest increases in risk by TC model and country, split by TC model

f, axes = plt.subplots(2, 2, figsize=(10, 8))
top_n = 10
rp = 50
tc_models = df.index.get_level_values("TC model").unique()

candidate_countries = set()
for tc_model in tc_models:
    at_rp = (
        df.loc[tc_model, :, rp - 1: rp + 1, :]
            .droplevel("TC model")
            .droplevel("return_period_years")["diff"]
    )
    top_increase = at_rp.sort_values(ascending=False).iloc[:top_n]
    top_decrease = at_rp.sort_values(ascending=True).iloc[:top_n]
    candidate_countries.update(top_increase.index)
    candidate_countries.update(top_decrease.index)

candidate_countries = sorted(candidate_countries)

# Map each country to a colour from tab20c
cmap = plt.get_cmap("tab20c", len(candidate_countries))
country_colors = {
    country: cmap(i) for i, country in enumerate(candidate_countries)
}

for tc_model, ax in zip(tc_models, axes.ravel()):
    to_plot = (
        df.loc[tc_model, :, rp - 1: rp + 1, :]
            .droplevel("TC model")
            .droplevel("return_period_years")
            .sort_values("diff", ascending=False)["diff"]
            .iloc[:top_n]
            .iloc[::-1]
    )
    colors = [country_colors[c] for c in to_plot.index]
    to_plot.plot(kind="barh", ax=ax, color=colors)
    ax.set_title(f"{tc_model}")
    ax.set_ylabel("")
    ax.set_xlim(0, 1.1 * df.loc[:, :, rp - 1: rp + 1]["diff"].max())
    ax.grid()
f.suptitle(f"Largest risk increase (at 1 in {rp} year people affected)")
f.supxlabel("Increase in population at risk (future - baseline)", y=0.1)
f.supylabel("Country")
plt.subplots_adjust(bottom=0.2, hspace=0.3)

In [ ]:
# Largest decrease in risk by TC model and country, split by TC model

f, axes = plt.subplots(2, 2, figsize=(10, 8))
top_n = 10
rp = 50
tc_models = df.index.get_level_values("TC model").unique()

for tc_model, ax in zip(tc_models, axes.ravel()):
    to_plot = (
        df.loc[tc_model, :, rp - 1: rp + 1, :]
            .droplevel("TC model")
            .droplevel("return_period_years")
            .sort_values("diff")["diff"]
            .iloc[:top_n]
            .iloc[::-1]
    )
    colors = [country_colors[c] for c in to_plot.index]
    to_plot.plot(kind="barh", ax=ax, color=colors)
    ax.set_title(f"{tc_model}")
    ax.set_ylabel("")
    ax.set_xlim(0, 1.1 * df.loc[:, :, rp - 1: rp + 1]["diff"].min())
    ax.grid()
f.suptitle(f"Largest risk decrease (at 1 in {rp} year people affected)")
f.supxlabel("Decrease in population at risk (future - baseline)", y=0.1)
f.supylabel("Country")
plt.subplots_adjust(bottom=0.2, hspace=0.3)